# Topic Modeling and Demographic Profiling

This notebook performs topic modeling with BERTopic on video summaries about anabolic steroids, followed by demographic analyses of users and content creators.

**Pipeline**: Preprocessing → Topic Modeling → Category Analysis (self-reports/symptoms) → Demographic Analysis (users and creators)

In [ ]:
# Cell 2: Library imports and definitions
import collections
import itertools
import json
import pickle as pkl
import random
import re
import ssl
import threading
import os
from scipy.stats import chi2_contingency, mannwhitneyu
from statsmodels.stats.multitest import multipletests

# Natural Language Processing (NLP)
import nltk
from nltk.corpus import stopwords
import spacy
from transformers import AutoTokenizer, AutoModel
import unidecode

# Topic modeling and data analysis
import pandas as pd
import numpy as np
from bertopic import BERTopic
from bertopic.representation import MaximalMarginalRelevance, OpenAI, KeyBERTInspired
from bertopic.vectorizers import ClassTfidfTransformer
from sentence_transformers import SentenceTransformer
from umap import UMAP
from hdbscan import HDBSCAN
from sklearn.feature_extraction.text import CountVectorizer

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud

# Other libraries
from tqdm.notebook import tqdm
import openai
import tiktoken

# Download NLTK resources (stopwords)
try:
    stopwords = stopwords.words('portuguese')
    stopwords.append('científicas')
except LookupError:
    nltk.download('stopwords')

# Custom class to ensure BERTopic model is serializable
class CustomBERTopic(BERTopic):
  def __getstate__(self):
      state = self.__dict__.copy()
      # Remove attributes that are not serializable
      # representation_model is removed because it may contain API clients (OpenAI)
      # which cannot be pickled.
      if 'representation_model' in state:
          del state['representation_model']
      return state

  def __setstate__(self, state):
      self.__dict__.update(state)
      
sns.color_palette('colorblind')

# Project paths and API key
PROJECT_ROOT = os.path.abspath(os.getcwd())
RESUMOS_DIR = os.path.join(PROJECT_ROOT, 'resumos')
# English alias for compatibility with refactored notebooks
SUMMARIES_DIR = RESUMOS_DIR
API_KEY_PATH = os.path.join(RESUMOS_DIR, 'api_key.txt')

if os.path.exists(API_KEY_PATH):
    with open(API_KEY_PATH, 'r') as k:
        key = k.read().splitlines()[0]
else:
    key = None

client = openai.OpenAI(api_key=key) if key else None

In [ ]:
# Cell 3: Data loading and preprocessing

# --- Input parameters ---
FILE_PATH_INPUT = os.path.join(RESUMOS_DIR, 'df_resumos_complete_cleaned.csv')
OUTPUT_DIR = os.path.join(RESUMOS_DIR, 'topic_analysis_final')
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Starting data loading and preprocessing...")

# Load the full dataset
# The engine will infer separators; specify sep=',' if needed
resumos_df = pd.read_csv(FILE_PATH_INPUT)
print(f"Original dataset shape: {resumos_df.shape}")

# --- Text cleaning functions ---
def remove_special_characters(text):
    text = str(text)
    # Remove newlines, tabs and multiple spaces
    return re.sub(r"(\\n)|(\\t)|(\\s{2,})", " ", text)

# Words that, if present, invalidate the summary
PALAVRAS_PARA_REMOVER = []  # list of forbidden words (kept name for compatibility)

def contem_palavras_proibidas(comentario, palavras_proibidas):
    """Check whether a text contains any forbidden words.

    Note: variable names are kept in Portuguese for compatibility with later cells.
    """
    comentario = str(comentario)
    for palavra in palavras_proibidas:
        if palavra.lower() in comentario.lower():
            return True
    return False

# --- Apply preprocessing ---
print("\nApplying cleaning and filters...")

# Ensure the 'resumo' column is string type
resumos_df['resumo'] = resumos_df['resumo'].astype(str)

resumos_df['resumo'] = resumos_df['resumo'].apply(remove_special_characters)
print(f"Shape after removing special characters: {resumos_df.shape}")

# Remove summaries containing prohibited words
resumos_df = resumos_df[~resumos_df['resumo'].apply(lambda x: contem_palavras_proibidas(x, PALAVRAS_PARA_REMOVER))]
print(f"Shape after removing prohibited words: {resumos_df.shape}")

# Remove very short summaries (less than 15 characters)
resumos_df = resumos_df[resumos_df["resumo"].str.len() >= 15]
print(f"Shape after removing short summaries: {resumos_df.shape}")

# Remove duplicate summaries to optimize model training
resumos_df.drop_duplicates(subset=['resumo'], inplace=True, keep='first')
print(f"Shape after deduplication: {resumos_df.shape}")

# Create documents list for BERTopic
docs = list(resumos_df['resumo'])
print(f"\nPreprocessing complete. {len(docs)} unique documents ready for modeling.")

---
## 1. Topic Modeling (BERTopic)

This section configures and trains a BERTopic model to identify topics in video summaries:
- **Preprocessing**: Remove special characters, duplicates and very short texts
- **BERTopic Pipeline**: BERT embeddings → UMAP (dimensionality reduction) → HDBSCAN (clustering) → c-TF-IDF (keyword extraction)
- **Refinement**: Outlier reduction, merging similar topics, and applying interpretable labels
- **Expected outcome**: a set of coherent topics describing themes in the corpus

In [ ]:
# Cell 4: Configure BERTopic model components

print("Configuring BERTopic components...")

# --- Tunable parameters ---
# Embedding Model: converts text into numeric vectors
EMBEDDING_MODEL = "rufimelo/bert-large-portuguese-cased-sts"
# EMBEDDING_MODEL = "neuralmind/bert-large-portuguese-cased"

# UMAP: dimensionality reduction
N_NEIGHBORS_UMAP = 15
N_COMPONENTS_UMAP = 10
MIN_DIST_UMAP = 0.1
METRIC_UMAP = 'cosine'

# HDBSCAN: clustering of vectors
MIN_CLUSTER_SIZE_HDBSCAN = 20
MIN_SAMPLES_HDBSCAN = 20
METRIC_HDBSCAN = 'euclidean'

# Vectorizer: vectorization for topic representation (c-TF-IDF)
MIN_DF_VECTORIZER = 0.001 # Ignore words that appear in less than this fraction of documents
MAX_DF_VECTORIZER = 0.95 # Ignore words that appear in more than this fraction of documents

# BERTopic: main model parameters
MIN_TOPIC_SIZE_BERTOPIC = 20
TOP_N_WORDS_BERTOPIC = 10
DIVERSITY_MMR = 0.6 # 0 = similar words, 1 = diverse words

# --- Initialize components ---

# 1. Embedding model
embedding_model = SentenceTransformer(EMBEDDING_MODEL)

# 2. Dimensionality reduction (UMAP)
umap_model = UMAP(
    n_neighbors=N_NEIGHBORS_UMAP,
    n_components=N_COMPONENTS_UMAP,
    min_dist=MIN_DIST_UMAP,
    metric=METRIC_UMAP,
    random_state=42,
)

# 3. Clustering model (HDBSCAN)
hdbscan_model = HDBSCAN(
    min_cluster_size=MIN_CLUSTER_SIZE_HDBSCAN,
    min_samples=MIN_SAMPLES_HDBSCAN,
    metric=METRIC_HDBSCAN,
    prediction_data=True,
)

# 4. Vectorizer (CountVectorizer)
vectorizer_model = CountVectorizer(
    min_df=MIN_DF_VECTORIZER,
    max_df=MAX_DF_VECTORIZER,
    stop_words=stopwords,
)

# 5. Term weighting (c-TF-IDF)
ctfidf_model = ClassTfidfTransformer(
    reduce_frequent_words=True,
)

# 6. Representation model (Maximal Marginal Relevance)
# Used to extract diverse and representative keywords
representation_model = {
    "KeyBERTInspired": KeyBERTInspired(),
    "MMR": MaximalMarginalRelevance(diversity=DIVERSITY_MMR),
}

# --- Assemble BERTopic model ---
model = CustomBERTopic(
    embedding_model=embedding_model,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    vectorizer_model=vectorizer_model,
    ctfidf_model=ctfidf_model,
    representation_model=representation_model,
    min_topic_size=MIN_TOPIC_SIZE_BERTOPIC,
    top_n_words=TOP_N_WORDS_BERTOPIC,
    verbose=True,
)

print("\nBERTopic configuration complete.")
print(model)

In [ ]:
# Cell 5: Model training and topic refinement

print("Starting model training; this may take a while.")

# Create a safe filename based on the embedding model name
model_safe_name = re.sub(r'[^a-zA-Z0-9_-]', '_', EMBEDDING_MODEL)
embeddings_file = os.path.join(OUTPUT_DIR, f"embeddings_{model_safe_name}.npy")

# Check if embeddings were already generated and saved
if os.path.exists(embeddings_file):
    print(f"Loading saved embeddings from '{embeddings_file}'...")
    embeddings = np.load(embeddings_file)
else:
    print("Generating embeddings and saving for future use...")
    embeddings = embedding_model.encode(docs, show_progress_bar=True)
    np.save(embeddings_file, embeddings)
    print(f"Embeddings saved to '{embeddings_file}'.")

# --- Training and transformation ---
topics, probabilities = model.fit_transform(docs, embeddings)

print(f"\nInitial training completed. Found {model.get_topic_info()['Topic'].nunique()} topics.")
print("Initial topic information:")
print(model.get_topic_info().head())

In [ ]:
# --- Post-processing and refinement ---

# 1. Outlier reduction
# Attempt to assign documents labeled as outliers (topic -1) to existing topics
print("\nStarting outlier reduction...")
new_topics = model.reduce_outliers(docs, topics, strategy="c-tf-idf", threshold=0.05)

# Update topics in the model
# The model will recompute representations based on the new assignments
model.update_topics(docs, topics=new_topics, vectorizer_model=vectorizer_model)
print(f"Outlier reduction complete. Current number of topics: {model.get_topic_info()['Topic'].nunique()}")

# --- Final results ---
print("\nFinal model trained and refined.")
final_topic_info = model.get_topic_info()
print(f"Total final topics: {final_topic_info['Topic'].nunique()}")
print("Final topics overview:")
print(final_topic_info)

In [ ]:
# 2. Reduce the number of topics
# Merge similar topics to reduce granularity if needed.
NR_TOPICS_FINAL = "auto"  # Set the desired final number of topics or None to skip reduction
if NR_TOPICS_FINAL is not None:
    print(f"\nReducing number of topics to {NR_TOPICS_FINAL}...")
    model.reduce_topics(docs, nr_topics=NR_TOPICS_FINAL)
    print("Topic reduction complete.")

    # # Reassign documents to the new topics after merging
    # topics, probs = model.transform(docs)
    # print("Topics reassigned after reduction.")

In [ ]:
# merging topics
topics_to_merge = [-1, 9]
model.merge_topics(docs, topics_to_merge)

In [ ]:
final_topic_info = model.get_topic_info()
topic_counts = final_topic_info.set_index('Topic')['Count']

for topic_id in final_topic_info['Topic']:
    if topic_id == -1:
        continue
    keywords = [w for w, _ in model.get_topic(topic_id)][:5]
    count = topic_counts.loc[topic_id]
    print(f"Topic ID: {topic_id}, Keywords: {keywords}, Count: {count}")

In [ ]:
# Applying custom labels to topics
new_labels = {
    0: "Personal Report and Informal Recommendation",
    1: "Oxandrolone: Uses and Indications",
    2: "Effects of Low Testosterone",
    3: "Replacement Therapy (TRT)",
    4: "Natural Testosterone Increase",
    5: "Cycles and Muscle Gain",
    6: "Effects of Trenbolone",
    7: "Technical Explanation of Testosterone",
    8: "Menopause and Testosterone in Women",
    9: "Personal Cycle Reports",
    10: "Cycle and PCT Discussion",
    11: "Hormonal Recovery and Fertility",
    12: "Authenticity Verification",
    13: "Pre-Hormonal Supplements",
    14: "Testosterone: Cardiovascular Risks",
    15: "Protocols for Weight Loss",
    16: "Steroids for Female Audience",
    17: "Exogenous Testosterone Dosages",
    18: "Female Steroid Reports",
    19: "Obesity and Testosterone",
    20: "Lower Body Training",
    21: "Aromatization and Estradiol Control",
    22: "Testosterone and Hair Loss",
    23: "Intramuscular Application Techniques",
    24: "Testosterone Physiology",
    25: "PCT and Hormonal Axis Restoration",
    26: "Testosterone and Prostate Health",
    27: "Phytotherapy and Testosterone Increase",
    28: "Trenbolone Neurotoxicity",
    29: "Hormone Therapy and Trans People",
    30: "Steroid Use in Adolescence"
}

# Apply the custom labels to the model
model.set_topic_labels(new_labels)

print("Custom topic labels successfully applied!")
print("\nPreviewing the new labels:")
print(model.get_topic_info()[['Topic', 'CustomName', 'Count']].head(10))

In [ ]:
# Cell 6: Assign topics and save artifacts

print("Assigning final topics to summaries and saving outputs...")

# Add final topic IDs to the original DataFrame
resumos_df['topic_id'] = model.topics_

# Map topic info (name, keywords) into the DataFrame
topic_info_df = model.get_topic_info()
final_df = pd.merge(
    resumos_df,
    topic_info_df[['Topic', 'Name', 'CustomName', 'Representation']],
    left_on='topic_id',
    right_on='Topic',
    how='left'
)
final_df.drop(columns=['Topic'], inplace=True)  # Remove duplicate 'Topic' column

# --- Save artifacts ---
# 1. Final BERTopic model
MODEL_PATH = os.path.join(OUTPUT_DIR, 'bertopic_model_final.pkl')
with open(MODEL_PATH, 'wb') as f:
    pkl.dump(model, f)
print(f"Final model saved to: {MODEL_PATH}")

# 2. DataFrame with topics
RESUMOS_PATH = os.path.join(OUTPUT_DIR, 'messages_with_topics_final.csv')
final_df.to_csv(RESUMOS_PATH, index=False, sep=';')
print(f"DataFrame with topics saved to: {RESUMOS_PATH}")

# 3. Topic descriptions
TOPIC_DESC_PATH = os.path.join(OUTPUT_DIR, 'topic_descriptions_final.csv')
topic_info_df.to_csv(TOPIC_DESC_PATH, index=False, sep=';')
print(f"Topic descriptions saved to: {TOPIC_DESC_PATH}")

In [ ]:
final_df = final_df[final_df['topic_id'] != -1]

df_videos_ = pd.read_csv('cleaned_data/filtered_videos.csv', usecols=['video_id', 'view_count', 'like_count'])
final_df_ = final_df.merge(
    df_videos_,
    on='video_id',
    how='inner'
)

engagement_by_topic = (
    final_df_.groupby('topic_id')
    .agg(
        n_videos=('video_id', 'nunique'),
        total_views=('view_count', 'sum'),
        total_likes=('like_count', 'sum')
    )
    .reset_index()
)

engagement_by_topic

In [ ]:
final_df['CustomName'] = final_df['topic_id'].astype(str) + ' - ' + final_df['CustomName']
topic_order = final_df.groupby('Name')['topic_id'].first().sort_values().index

In [ ]:
# --- Bar Chart: Topic Distribution ---
plt.figure(figsize=(12, 6))
sns.countplot(y='Name', data=final_df, order=topic_order, palette='viridis')

plt.xlabel('# of Videos')
plt.ylabel('Topic')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'topic_distribution_bar_chart_en.pdf'))
plt.show()

In [ ]:
for i in range(final_df['topic_id'].nunique()):
    print('Topic ID: ', i)
    print(model.get_representative_docs(i))
    print('\n')

In [ ]:
df = pd.read_csv('modelo/prediction/final_comments_match_cleaned_pred.csv')
df_full = final_df.merge(df, on='video_id', how='left')
df_full.head()

In [ ]:
topics_metric = df_full.groupby('topic_id').agg(
    n_comments=('comment_id', 'nunique'),
    n_users=('author_channel_id', 'nunique'),
    n_channels=('channel_id', 'nunique'),
    n_replies=('is_reply', 'sum')
).reset_index()

topics_metric

---
## 2. Category Analysis by Topic

This section merges topic assignments with comment classifications (self-report and symptoms/side effects) produced by the ML model. It computes and visualizes the proportion of each category per topic and compares them to global averages to identify topics with atypical concentrations.

In [ ]:
df_full.to_csv('./resumos/df_full.csv', index=False)

In [ ]:
df.shape, df_full.shape

In [ ]:
print(f"Number of unique comments: {df_full['comment_id'].nunique()}")
print(f"Number of unique users: {df_full['author_channel_id'].nunique()}")
print(f"Number of unique videos: {df_full['video_id'].nunique()}")
print(f"Number of unique channels: {df_full['channel_id'].nunique()}")

In [ ]:
def calcular_proporcoes(df_grupo):
    total_grupo = len(df_grupo)
    
    is_autorrelato = df_grupo['pred_Autorrelato'] == 1
    is_sintoma = df_grupo['pred_Sintomas_Efeitos_Colaterais'] == 1
    
    return pd.Series({
        'porc_Autorrelato': len(df_grupo[is_autorrelato & ~is_sintoma]) / total_grupo * 100,
        'porc_Sintomas':    len(df_grupo[~is_autorrelato & is_sintoma]) / total_grupo * 100,
        'porc_Ambos':       len(df_grupo[is_autorrelato & is_sintoma]) / total_grupo * 100,
        'porc_NA':          len(df_grupo[~is_autorrelato & ~is_sintoma]) / total_grupo * 100,
        'n_docs':           total_grupo
    })

In [ ]:
cols = ['pred_Autorrelato', 'pred_Sintomas_Efeitos_Colaterais']
porc_cat = (
    df_full.groupby('topic_id')[cols]
    .apply(calcular_proporcoes)
    .reset_index()
)

In [ ]:
porc_total = calcular_proporcoes(df)

print(f"Global percentage of self-report: {porc_total['porc_Autorrelato']:.2f}%")
print(f"Global percentage of symptoms/side-effects: {porc_total['porc_Sintomas']:.2f}%")
print(f"Global percentage of both: {porc_total['porc_Ambos']:.2f}%")
print(f"Global percentage of NA: {porc_total['porc_NA']:.2f}%")

In [ ]:
cols_interesse = ['porc_Autorrelato', 'porc_Sintomas', 'porc_Ambos']

df_plot_cat = porc_cat.melt(
    id_vars='topic_id',
    value_vars=cols_interesse,
    var_name='Categoria',
    value_name='Porcentagem'
)

df_plot_cat['Categoria'] = df_plot_cat['Categoria'].str.replace('porc_', '')
df_plot_cat['Categoria'] = df_plot_cat['Categoria'].replace({
    'Autorrelato': 'Self-report',
    'Sintomas': 'Symptoms / Side Effects',
    'Ambos': 'Both'
})
palette_colors = {
    'Self-report': "#1f77b4",
    'Symptoms / Side Effects': '#ff7f0e',
    'Both': '#2ca02c',
    'NA': '#7a7a7a'
}

plt.figure(figsize=(10, 4))

sns.barplot(
    data=df_plot_cat,
    x='topic_id',
    y='Porcentagem',
    hue='Categoria',
    palette=palette_colors,
    edgecolor='white'
)

for col_original, nome_limpo in zip(cols_interesse, palette_colors.keys()):
    valor_global = porc_total[col_original]
    cor = palette_colors[nome_limpo]
    
    plt.axhline(
        y=valor_global, 
        color=cor, 
        linestyle='--', 
    )

plt.ylabel("% of Comments")
plt.xlabel("Topic ID")
plt.legend(loc='upper center', ncol=4)

plt.tight_layout()
# plt.savefig(os.path.join(OUTPUT_DIR, 'barras_classes_topicos.pdf'))
plt.show()

In [ ]:
df_attr = pd.read_csv("profile_files/user_atribbutes_unico.csv")

df_full_attr = df_attr.merge(
    df_full,
    left_on="id",
    right_on="author_channel_id",
    how="inner"
).drop_duplicates(subset=['id', 'topic_id']).reset_index(drop=True)

---
## 3. User Demographics Analysis (Commenters)

Analyzes age and gender of users who comment on videos for each topic:
- **Data**: User attributes extracted from profile face analysis
- **Visualizations**: Age boxplots and gender bar charts per topic
- **Statistical tests**: Mann-Whitney U (age) and Chi-square (gender) with Bonferroni correction to find topics with demographic profiles significantly different from the global average

In [ ]:
df_full_attr.shape

In [ ]:
plt.figure(figsize=(10, 4))

sns.boxplot(
    df_full_attr,
    x='topic_id',
    y='age',
    color='#1f77b4',
    showfliers=False,
    linecolor='#4d4d4d'
)

plt.ylabel('Age')
plt.xlabel('Topic ID')
plt.tight_layout()
# plt.savefig(os.path.join(OUTPUT_DIR, 'boxplot_idades_topicos.pdf'))
plt.show()

In [ ]:
print(f"Mean age: {df_full_attr['age'].mean():.2f} +- {df_full_attr['age'].std():.2f} years")
print(f"Median age: {df_full_attr['age'].median()} years")

In [ ]:
for topic_id in df_full_attr['topic_id'].unique():
    df_interest = df_full_attr[df_full_attr['topic_id'] == topic_id]['age']
    df_general = df_full_attr[df_full_attr['topic_id'] != topic_id]['age']

    stat, p = mannwhitneyu(df_interest, df_general, alternative='two-sided')

    alpha_corrected = 0.05 / df_full_attr['topic_id'].nunique()

    if p < alpha_corrected:
        median_topic = df_interest.median()
        median_general = df_general.median()
        direction = "higher" if median_topic > median_general else "lower"
        print(f'Topic {topic_id}: Age {direction} than global (Med: {median_topic:.1f} vs {median_general:.1f}, p={p:.4f})')

In [ ]:
gender_counts = df_full_attr.groupby(['topic_id', 'gender']).size().unstack(fill_value=0)
gender_pct = gender_counts.div(gender_counts.sum(axis=1), axis=0) * 100

df_plot = gender_pct.reset_index().melt(id_vars='topic_id', value_name='percentage', var_name='gender')
df_plot['gender'] = df_plot['gender'].replace({'F': 'Female', 'M': 'Male'})

plt.figure(figsize=(10, 4))

ax = sns.barplot(
    data=df_plot,
    x='topic_id',
    y='percentage',
    hue='gender',
    palette={'Female': '#ff7f0e', 'Male': '#1f77b4'},
    edgecolor='white'
)

counts_total = df_full_attr['gender'].value_counts()
pct_w_global = (counts_total['F'] / counts_total.sum()) * 100
pct_m_global = (counts_total['M'] / counts_total.sum()) * 100

plt.axhline(y=pct_w_global, color='#ff7f0e', linestyle='--')
plt.axhline(y=pct_m_global, color='#1f77b4', linestyle='--')

plt.ylabel("Gender (%)")
plt.xlabel("Topic ID")
plt.legend(loc='best', ncol=2)
plt.ylim(0, 100)

plt.tight_layout()
# plt.savefig(os.path.join(OUTPUT_DIR, 'barras_gender_topicos.pdf'))
plt.show()

In [ ]:
print(f"Global percentage of male: {pct_m_global:.2f}")
print(f"Global percentage of female: {pct_w_global:.2f}")

In [ ]:
for topic_id in df_full_attr['topic_id'].unique():
    target_topic = df_full_attr[df_full_attr['topic_id'] == topic_id]['gender'].value_counts().reindex(['M', 'F'], fill_value=0)
    other_topics = df_full_attr[df_full_attr['topic_id'] != topic_id]['gender'].value_counts().reindex(['M', 'F'], fill_value=0)
    
    tabela = [target_topic.values, other_topics.values]
    
    chi2, p, dof, expected = chi2_contingency(tabela)
    
    if p < (0.05 / df_full_attr['topic_id'].nunique()):
        print(f"Topic {topic_id}: gender proportion is significantly different from global (p={p:.4f})")

In [ ]:
df_videos = pd.read_csv('videos_files/videos_demographics_complete.csv')
df_videos = df_videos[(df_videos['age'] >= 15)]

df_channels = pd.read_csv('cleaned_data/filtered_videos.csv', usecols=['video_id', 'channel_id'])

df_videos_full_attr = df_videos.merge(
    final_df,
    on='video_id',
    how='inner'
)

df_videos_full_attr = df_videos_full_attr.merge(
    df_channels,
    on='video_id',
    how='left'
)
print(df_videos_full_attr.shape)

---
## 4. Creator Demographics Analysis (Videos)

Analyzes demographic characteristics of content creators extracted from video frames:
- **Data**: Age, gender, and gender consistency across the video
- **Gender consistency**: A video is considered consistent if the top-5 frames of the presenter receive the same gender label
- **Analyses**: Distribution of creator age/gender by topic, identification of topics with higher inconsistency (interviews, panels), and analysis of channels with multiple presenters

In [ ]:
print(f"Average frame magnitude: {round(df_videos_full_attr['quality_magnitude'].mean(), 2)}")

In [ ]:
plt.figure(figsize=(10, 4))

sns.boxplot(
    df_videos_full_attr,
    x='topic_id',
    y='age',
    color='#1f77b4',
    showfliers=False,
    linecolor='#4d4d4d'
)

plt.ylabel('Age')
plt.xlabel('Topic ID')
plt.tight_layout()
# plt.savefig(os.path.join(OUTPUT_DIR, 'videos_boxplot_idades_topicos.pdf'))
plt.show()

In [ ]:
print(f"Mean age (videos): {df_videos_full_attr['age'].mean():.2f} +- {df_videos_full_attr['age'].std():.2f} years")
print(f"Median age (videos): {df_videos_full_attr['age'].median()} years")

In [ ]:
for topic_id in df_videos_full_attr['topic_id'].unique():
    df_interest = df_videos_full_attr[df_videos_full_attr['topic_id'] == topic_id]['age']
    df_geral = df_videos_full_attr[df_videos_full_attr['topic_id'] != topic_id]['age']
    
    stat, p = mannwhitneyu(df_interest, df_geral, alternative='two-sided')
    
    alpha_corrigido = 0.05 / df_videos_full_attr['topic_id'].nunique()
    
    if p < alpha_corrigido:
        mediana_topic = df_interest.median()
        mediana_geral = df_geral.median()
        direcao = "higher" if mediana_topic > mediana_geral else "lower"
        print(f'Topic {topic_id}: Age {direcao} than global (Med: {mediana_topic:.1f} vs {mediana_geral:.1f}, p={p:.4f})')

In [ ]:
gender_counts = df_videos_full_attr.groupby(['topic_id', 'gender']).size().unstack(fill_value=0)
gender_pct = gender_counts.div(gender_counts.sum(axis=1), axis=0) * 100

df_plot = gender_pct.reset_index().melt(id_vars='topic_id', value_name='percentage', var_name='gender')
df_plot['gender'] = df_plot['gender'].replace({'F': 'Female', 'M': 'Male'})

plt.figure(figsize=(10, 4))

ax = sns.barplot(
    data=df_plot,
    x='topic_id',
    y='percentage',
    hue='gender',
    palette={'Female': '#ff7f0e', 'Male': '#1f77b4'},
    edgecolor='white'
)

counts_total = df_videos_full_attr['gender'].value_counts()
pct_w_global = (counts_total['F'] / counts_total.sum()) * 100
pct_m_global = (counts_total['M'] / counts_total.sum()) * 100

plt.axhline(y=pct_w_global, color='#ff7f0e', linestyle='--')
plt.axhline(y=pct_m_global, color='#1f77b4', linestyle='--')

plt.ylabel("Gender (%)")
plt.xlabel("Topic ID")
plt.legend(loc='best', ncol=2)
plt.ylim(0, 100)

plt.tight_layout()
# plt.savefig(os.path.join(OUTPUT_DIR, 'videos_barras_gender_topicos.pdf'))
plt.show()

In [ ]:
print(f"Global percentage of male (videos): {pct_m_global:.2f}")
print(f"Global percentage of female (videos): {pct_w_global:.2f}")

In [ ]:
for topic_id in df_videos_full_attr['topic_id'].unique():
    target_topic = df_videos_full_attr[df_videos_full_attr['topic_id'] == topic_id]['gender'].value_counts().reindex(['M', 'F'], fill_value=0)
    other_topics = df_videos_full_attr[df_videos_full_attr['topic_id'] != topic_id]['gender'].value_counts().reindex(['M', 'F'], fill_value=0)
    
    tabela = [target_topic.values, other_topics.values]
    
    chi2, p, dof, expected = chi2_contingency(tabela)
        
    if p < (0.05 / df_videos_full_attr['topic_id'].nunique()):
        print(f"Topic {topic_id}: gender proportion is significantly different from global (p={p:.4f})")

In [ ]:
cluster_percentage = round(df_videos_full_attr['cluster_percentage'].mean(), 2)
mean_std = round(df_videos_full_attr['age_std'].mean(), 2)

print(f'Mean: the largest cluster represents {cluster_percentage}% of faces on average')
print(f'Mean age std: {mean_std} years')

In [ ]:
print(f"Percentage of videos with gender consistency: {df_videos_full_attr[df_videos_full_attr['gender_consistency'] == 1].shape[0] / df_videos_full_attr.shape[0]:.2f}")

In [ ]:
# Proportion of gender_consistency < 1 per topic
consistency_by_topic = (
    df_videos_full_attr.groupby('topic_id')
    .agg(
        prop_inconsistent=('gender_consistency', lambda x: (x < 1).mean() * 100),
        n_videos=('gender_consistency', 'count')
    )
    .reset_index()
)

In [ ]:
prop_global = (df_videos_full_attr['gender_consistency'] < 1).mean() * 100

plt.figure(figsize=(10, 4))

sns.barplot(
    data=consistency_by_topic,
    x='topic_id',
    y='prop_inconsistent',
    color='#1f77b4',
    edgecolor='white'
)

plt.axhline(
    y=prop_global,
    linestyle='--',
    color='#7a7a7a',
    label=f'Global: {prop_global:.1f}%'
)

plt.ylabel("Gender Inconsistency (%)")
plt.xlabel("Topic ID")
plt.legend(loc='best')
plt.tight_layout()
# plt.savefig(os.path.join(OUTPUT_DIR, 'videos_gender_consistency_topicos.pdf'))
plt.show()

In [ ]:
quality_by_topic = (
    df_videos_full_attr.groupby('topic_id')
    .agg(mean_quality=('quality_magnitude', 'mean'))
    .reset_index()
)

quality_global = df_videos_full_attr['quality_magnitude'].mean()

plt.figure(figsize=(10, 4))

sns.barplot(
    data=quality_by_topic,
    x='topic_id',
    y='mean_quality',
    color='#1f77b4',
    edgecolor='white'
)

plt.axhline(
    y=quality_global,
    linestyle='--',
    color='#7a7a7a',
    label=f'Global: {quality_global:.2f}'
)

plt.ylabel("Frame Quality (mean magnitude)")
plt.xlabel("Topic ID")
plt.legend(loc='best')
plt.tight_layout()
# plt.savefig(os.path.join(OUTPUT_DIR, 'videos_frame_quality_topicos.pdf'))
plt.show()

In [ ]:
df_videos_full_attr['is_inconsistent'] = df_videos_full_attr['gender_consistency'] < 1

for topic_id in df_videos_full_attr['topic_id'].unique():
    target_topic = df_videos_full_attr[df_videos_full_attr['topic_id'] == topic_id]['is_inconsistent'].value_counts().reindex([False, True], fill_value=0)
    other_topics = df_videos_full_attr[df_videos_full_attr['topic_id'] != topic_id]['is_inconsistent'].value_counts().reindex([False, True], fill_value=0)
    
    tabela = [target_topic.values, other_topics.values]
    
    chi2, p, dof, expected = chi2_contingency(tabela)
    
    alpha_corrigido = 0.05 / df_videos_full_attr['topic_id'].nunique()
    
    if p < alpha_corrigido:
        print(f"Topic {topic_id}: proportion of inconsistencies is significantly different from global (p={p:.4f})")

In [ ]:
canais_incons = df_videos_full_attr[df_videos_full_attr['is_inconsistent']].groupby('topic_id')['channel_id'].nunique()
total_canais = df_videos_full_attr.groupby('topic_id')['channel_id'].nunique()

df_canais_incons = pd.DataFrame({
    'topic_id': total_canais.index,
    'inconsistent_channels': canais_incons.reindex(total_canais.index, fill_value=0).values,
    'total_channels': total_canais.values,
    'proportion': (canais_incons.reindex(total_canais.index, fill_value=0) / total_canais * 100).round(2).values
})

canais_incons_global = df_videos_full_attr[df_videos_full_attr['is_inconsistent']]['channel_id'].nunique()
total_canais_global = df_videos_full_attr['channel_id'].nunique()
prop_canais_incons_global = round(canais_incons_global / total_canais_global * 100, 2)

print(f"Global proportion: {prop_canais_incons_global}% of channels ({canais_incons_global}/{total_canais_global})")

df_canais_incons